In [1]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 34.0 MB/s eta 0:00:00


In [2]:
!pip show torch transformers datasets peft bitsandbytes accelerate

Name: torch
Version: 2.8.0+cu126
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, bitsandbytes, easyocr, fastai, kornia, peft, pytorch-ignite, pytorch-lightning, sentence-transformers, stable-baselines3, timm, torchaudio, torchdata, torchmetrics, torchvision
---
Name: transformers
Version: 4.57.1
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https:

In [3]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, PeftModel


2026-02-09 08:07:42.288188: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770624462.512092      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770624462.574517      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770624463.123400      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770624463.123441      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770624463.123443      24 computation_placer.cc:177] computation placer alr

In [4]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# CHANGE this to your Kaggle dataset path
DATA_PATH = "/kaggle/input/eye-disease-train-v2/eye_disease_train_v2.jsonl"

OUTPUT_DIR = "/kaggle/working/eye_llm_lora"


Tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# REQUIRED for causal LM + Trainer padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Load dataset

In [6]:
dataset = load_dataset("json", data_files=DATA_PATH)["train"]
print("Dataset size:", len(dataset))

Generating train split: 0 examples [00:00, ? examples/s]

Dataset size: 1012


Apply chat template

In [7]:
def format_chat(sample):
    messages = [
        {
            "role": "system",
            "content": "You are a medical assistant specialized in eye diseases."
        },
        {
            "role": "user",
            "content": f"{sample['instruction']}\n{sample['input']}"
        },
        {
            "role": "assistant",
            "content": sample["output"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    enc = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding="max_length"
    )

    enc["labels"] = enc["input_ids"].copy()
    return enc


dataset = dataset.map(
    format_chat,
    remove_columns=dataset.column_names
)


Map:   0%|          | 0/1012 [00:00<?, ? examples/s]

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Attach LoRA

In [9]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


TrainingArguments

In [10]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,  # IMPORTANT for causal LM
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()


Step,Training Loss
10,12.642900
20,3.984400
30,0.462200
40,0.380000
50,0.352800
60,0.300300
70,0.259100
80,0.261300
90,0.235500
100,0.235200


TrainOutput(global_step=381, training_loss=0.6610149396216776, metrics={'train_runtime': 701.8279, 'train_samples_per_second': 4.326, 'train_steps_per_second': 0.543, 'total_flos': 9658980397744128.0, 'train_loss': 0.6610149396216776, 'epoch': 3.0})

Save

In [11]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("LoRA adapter saved to:", OUTPUT_DIR)


LoRA adapter saved to: /kaggle/working/eye_llm_lora


Reload fine-tuned model for inference

In [12]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear

In [13]:
prompt = (
    "Patient is female, 64 years old, with a history of hypertension. "
    "What general medical advice can be provided?"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Patient is female, 64 years old, with a history of hypertension. What general medical advice can be provided?


In [14]:
messages = [
    {
        "role": "system",
        "content":"You are a medical assistant specialized in eye diseases. "
    },
    {
        "role": "user",
        "content": (
            "I am  female, 64 years old, with a history of hypertension."
            "Provide general medical advice for me."
        )
    }
]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10000,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        eos_token_id = tokenizer.eos_token_id,
        pad_token_id = tokenizer.eos_token_id,

    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


<|system|>
You are a medical assistant specialized in eye diseases.  
<|user|>
I am  female, 64 years old, with a history of hypertension.Provide general medical advice for me. 
<|assistant|>
Hypertension can cause ocular complications like glaucoma, cataracts, and peripheral veins damage, all of which can worsen in the setting of secondary hypertension. You are advised to closely monitor your blood pressure readings and take medication to manage your hypertension.


In [15]:
# Install transformers from source - only needed for versions <= v4.34
# pip install git+https://github.com/huggingface/transformers.git
# pip install accelerate
import torch

from transformers import pipeline,AutoTokenizer
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# CHANGE this to your Kaggle dataset path
DATA_PATH = "/kaggle/input/eye-disease-train/eye_disease_train.jsonl"

OUTPUT_DIR = "/kaggle/working/eye_llm_lora"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
pipe_v1 = pipeline("text-generation", model=MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")

# We use the tokenizer's chat template to format each message - see https://huggingface.co/docs/transformers/main/en/chat_templating
messages_1 = [
    {
        "role": "system",
        "content": "You are a medical assistant specialized in eye diseases.",
    },
    {"role": "user",         
     "content": (
            "I am  female, 64 years old, with a history of hypertension. "
            "Provide general medical advice for me"
        )},
]
prompt_v1 = pipe_v1.tokenizer.apply_chat_template(messages_1, tokenize=False, add_generation_prompt=True)
outputs_v1 = pipe_v1(prompt_v1, max_new_tokens=10000, do_sample=True, temperature=0.7, top_k=50, top_p=0.95,        eos_token_id = tokenizer.eos_token_id,
        pad_token_id = tokenizer.eos_token_id,)
print(outputs_v1[0]["generated_text"])
# <|system|>
# You are a friendly chatbot who always responds in the style of a pirate.</s>
# <|user|>
# How many helicopters can a human eat in one sitting?</s>
# <|assistant|>
# ...


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


<|system|>
You are a medical assistant specialized in eye diseases.</s>
<|user|>
I am  female, 64 years old, with a history of hypertension. Provide general medical advice for me</s>
<|assistant|>
Dear Patient,

I'm sorry to hear that you have hypertension. Hypertension is a common condition that causes your blood pressure to be elevated, which can lead to several health problems. Here are some general medical advice that you can follow:

1. Lifestyle changes: To manage your hypertension, it's essential to make lifestyle changes. Exercise regularly, maintain a healthy weight, limit your alcohol consumption, and quit smoking.

2. Medications: Your doctor may prescribe medications such as aspirin, beta blockers, calcium channel blockers, or angiotensin-converting enzyme (ACE) inhibitors to help lower your blood pressure.

3. Dietary changes: A healthy diet that includes plenty of fruits, vegetables, whole grains, lean proteins, and low-fat dairy products can help you maintain a healthy w

In [ ]:
import torch
from threading import Thread
from transformers import pipeline, AutoTokenizer, TextIteratorStreamer

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    tokenizer=tokenizer,
)

messages = [
    {"role": "system", "content": "You are a friendly chatbot who always responds in the style of a pirate"},
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
]
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Run generation in background thread
generation_kwargs = dict(
    text_inputs=prompt, max_new_tokens=256, do_sample=True,
    temperature=0.7, top_k=50, top_p=0.95, streamer=streamer
)
thread = Thread(target=pipe, kwargs=generation_kwargs)
thread.start()

# Consume tokens as they arrive
for token in streamer:
    print(token, end="", flush=True)

thread.join()